In [46]:
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import spacy

In [27]:
metadata_df = pd.read_csv("../data/manifestos/metadata.csv", index_col="id")

In [28]:
metadata_df.info()

<class 'pandas.DataFrame'>
Index: 12521 entries, EL123_P_1981_001 to EL198_L_1993_03_095_09_2_PF_02
Data columns (total 41 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   date                            12521 non-null  str    
 1   subject                         12521 non-null  str    
 2   title                           12521 non-null  str    
 3   contexte-election               12521 non-null  str    
 4   contexte-tour                   12521 non-null  int64  
 5   cote                            12521 non-null  str    
 6   departement                     12498 non-null  str    
 7   departement-nom                 12498 non-null  str    
 8   departement-insee               12498 non-null  str    
 9   identifiant de circonscription  12498 non-null  float64
 10  images                          12510 non-null  str    
 11  pdf                             12486 non-null  str    
 12  ocr_url 

In [ ]:
def map_political_blocs_temporal(row):
    label = str(row["titulaire-soutien"]).lower()
    year = row["date"].year

    if pd.isna(label) or label == "non mentionné":
        return "UNKNOWN"

    # 1. Left and Far-Right (Stable across 1981-1993)
    if "communiste français" in label:
        return "LEFT_PCF"
    if "front national" in label:
        return "FAR_RIGHT_FN"
    if any(
        x in label for x in ["parti socialiste", "socialiste", "radicaux de gauche"]
    ):
        return "LEFT_PS"
    if any(x in label for x in ["verts", "écologie", "écologiste"]):
        return "ECO"
    if any(x in label for x in ["lutte ouvrière", "ligue communiste"]):
        return "FAR_LEFT"

    # 2. The Right-Wing Bloc (Time-Conditional)
    is_rpr = "rassemblement pour la république" in label
    is_udf = "union pour la démocratie française" in label

    if is_rpr and is_udf:
        # Explicit alliances
        if year == "1981":
            return "RIGHT_UNM"  # Union pour la nouvelle majorité
        if year == "1988":
            return "RIGHT_URC"  # Union du Rassemblement et du Centre
        if year == "1993":
            return "RIGHT_UPF"  # Union pour la France
    elif is_rpr:
        return "RIGHT_RPR"
    elif is_udf:
        return "RIGHT_UDF"

    return "OTHER"

In [ ]:
metadata_df["date"] = pd.to_datetime(metadata_df["date"])
metadata_df["target_label"] = metadata_df.apply(map_political_blocs_temporal, axis=1)
main_parties_metadata_df = metadata_df[
    ~metadata_df["target_label"].str.contains("UNKNOWN|OTHER")
]
main_parties_metadata_df = main_parties_metadata_df[
    main_parties_metadata_df["contexte-election"] == "législatives"
]
main_parties_metadata_df.groupby("target_label")["target_label"].count()

target_label
ECO             1184
FAR_LEFT         497
FAR_RIGHT_FN    1242
LEFT_PCF        1605
LEFT_PS         1563
RIGHT_RPR        271
RIGHT_UDF        254
Name: target_label, dtype: int64

In [ ]:
main_parties_metadata_df.assign(year=main_parties_metadata_df["date"].dt.year).groupby(
    "year"
).count()

,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,identifiant de circonscription,...,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,target_label
year,,,,,,,,,,,,,,,,,,,,,
1981,1997,1997,1997,1997,1997,1997,1997,1997,1997,1997,...,1997,1997,1997,1997,1997,1997,1997,1997,1997,1997
1988,1434,1434,1434,1434,1434,1434,1434,1434,1434,1434,...,1434,1434,1434,1434,1434,1434,1434,1434,1434,1434
1993,3185,3185,3185,3185,3185,3185,3185,3185,3185,3185,...,3185,3185,3185,3185,3185,3185,3185,3185,3185,3185


In [42]:
main_parties_metadata_df

,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,identifiant de circonscription,...,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,target_label
id,,,,,,,,,,,,,,,,,,,,,
EL134_L_1981_06_001_01_1_PF_01,1981-06-14,France;Assemblée Nationale;Ve République;Élect...,"Élections législatives de 1981, Ain - 01, circ...",législatives,1,EL134,01,Ain,01 - Ain,1.0,...,non mentionné,infirmière,non mentionné,non mentionné,syndicat,non mentionné,Parti socialiste unifié,non mentionné,non,LEFT_PS
EL134_L_1981_06_001_01_1_PF_02,1981-06-14,Élections législatives;Ve République;Assemblée...,"Élections législatives de 1981, Ain - 01, circ...",législatives,1,EL134,01,Ain,01 - Ain,1.0,...,non mentionné,non mentionné,maire-adjoint,non mentionné,syndicat,non mentionné,Parti communiste français,Union de la gauche,non,LEFT_PCF
EL134_L_1981_06_001_01_1_PF_03,1981-06-14,Élections législatives;Ve République;France;As...,"Élections législatives de 1981, Ain - 01, circ...",législatives,1,EL134,01,Ain,01 - Ain,1.0,...,non mentionné,docteur,non mentionné,non mentionné,non mentionné,non mentionné,Parti socialiste;Mouvement des radicaux de gauche,non mentionné,non,LEFT_PS
EL134_L_1981_06_001_01_1_PF_04,1981-06-14,Assemblée Nationale;France;Ve République;Élect...,"Élections législatives de 1981, Ain - 01, circ...",législatives,1,EL134,01,Ain,01 - Ain,1.0,...,non mentionné,commerçant,non mentionné,non mentionné,culture;politique,non mentionné,Fédération des socialistes démocrates,non mentionné,non,LEFT_PS
EL134_L_1981_06_001_02_1_PF_02,1981-06-14,France;Élections législatives;Ve République;As...,"Élections législatives de 1981, Ain - 01, circ...",législatives,1,EL134,01,Ain,01 - Ain,2.0,...,entre 50 et 59 ans,cadre supérieur SNCF,maire,non mentionné,non mentionné,non mentionné,Parti communiste français,Union de la gauche,non,LEFT_PCF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
EL198_L_1993_03_095_03_2_PF_02,1993-03-28,Ve République;Élections législatives;Assemblée...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,2,EL198,95,Val-d'Oise,95 - Val-d'Oise,3.0,...,non mentionné,non mentionné,conseiller général;maire,non mentionné,non mentionné,non mentionné,Union pour la démocratie française,Opposition,non,RIGHT_UDF
EL198_L_1993_03_095_04_2_PF_02,1993-03-28,France;Assemblée Nationale;Ve République;Élect...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,2,EL198,95,Val-d'Oise,95 - Val-d'Oise,4.0,...,entre 60 et 69 ans,non mentionné,conseiller municipal,non mentionné,non mentionné,retraité,Front national,non mentionné,non,FAR_RIGHT_FN
EL198_L_1993_03_095_06_2_PF_02,1993-03-28,Ve République;Assemblée Nationale;France;Élect...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,2,EL198,95,Val-d'Oise,95 - Val-d'Oise,6.0,...,entre 50 et 59 ans,cadre commercial,conseiller municipal,non mentionné,politique,non mentionné,Front national,non mentionné,non,FAR_RIGHT_FN


In [ ]:
districts = pd.read_csv("../data/gazetteers/v_arrondissement_2025.csv")["LIBELLE"]
cities = pd.read_csv("../data/gazetteers/v_commune_2025.csv")["LIBELLE"]
overseas_cities = pd.read_csv("../data/gazetteers/v_commune_comer_2025.csv")["LIBELLE"]
departments = pd.read_csv("../data/gazetteers/v_departement_2025.csv")["LIBELLE"]
regions = pd.read_csv("../data/gazetteers/v_region_2025.csv")["LIBELLE"]
overseas_regions = pd.read_csv("../data/gazetteers/v_comer_2025.csv")["LIBELLE"]
raw_gazetteer = pd.concat(
    [districts, cities, overseas_cities, departments, regions, overseas_regions]
).dropna()

In [ ]:
def strip_accents(text: str) -> str:
    """Normalize diacritics to mitigate OCR-induced accent-dropping."""
    return "".join(
        c
        for c in unicodedata.normalize("NFKD", text)
        if unicodedata.category(c) != "Mn"
    )


In [ ]:
from typing import List, Set


def build_rigorous_gazetteer(file_paths: List[str]) -> Set[str]:
    """
    Constructs a deterministic geographical exclusion set by aggregating
    INSEE COG administrative files while protecting overlapping political ontology.
    """
    # Encapsulated Semantic Safeguards
    POLITICAL_SAFE_WORDS = {
        "nouvelle",
        "nouveau",
        "grand",
        "grande",
        "centre",
        "pays",
        "nord",
        "sud",
        "est",
        "ouest",
        "union",
        "val",
        "haute",
        "hauts",
        "bas",
    }

    # 1. Memory-efficient concatenated loading
    series_list = []
    for path in file_paths:
        # Strictly load the required column to minimize RAM overhead
        try:
            df = pd.read_csv(path, usecols=["LIBELLE"])
            series_list.append(df["LIBELLE"])
        except ValueError:
            raise RuntimeError(f"Column 'LIBELLE' not found in {path}")

    # Combine and drop nulls
    raw_gazetteer = pd.concat(series_list, ignore_index=True).dropna()

    gazetteer = set()

    # 2. Compile regex strictly ONCE outside the loop for O(N) efficiency
    tokenizer = re.compile(r"\b[a-z]{3,}\b")

    # 3. Extract and normalize
    for name in raw_gazetteer:
        # Assumes strip_accents() is defined in your global scope
        norm_name = strip_accents(name.lower())
        tokens = tokenizer.findall(norm_name)

        for token in tokens:
            if token not in POLITICAL_SAFE_WORDS:
                gazetteer.add(token)

    return gazetteer

In [ ]:
insee_files = [
    "../data/gazetteers/v_arrondissement_2025.csv",
    "../data/gazetteers/v_commune_2025.csv",
    "../data/gazetteers/v_commune_comer_2025.csv",
    "../data/gazetteers/v_departement_2025.csv",
    "../data/gazetteers/v_region_2025.csv",
    "../data/gazetteers/v_comer_2025.csv",
]
GAZETTEER = build_rigorous_gazetteer(insee_files)

In [ ]:
try:
    nlp = spacy.load("fr_core_news_lg", disable=["parser"])
except OSError:
    raise RuntimeError("Run: python -m spacy download fr_core_news_lg")

DISCOURSE_MARKERS = {
    "madame",
    "monsieur",
    "mademoiselle",
    "électrice",
    "électeur",
    "concitoyen",
    "cher",
    "chère",
    "salutation",
    "dévoué",
    "fidèle",
    "candidat",
    "circonscription",
    "vote",
    "voter",
    "élection",
    "scrutin",
    "tour",
    "premier",
    "deuxième",
    "suppléant",
    "bulletin",
    "suffrage",
    "député",
    "canton",
    "maire",
    "sortant",
    "mandat",
    "assemblée",
    "vouloir",
    "pouvoir",
    "devoir",
    "faire",
    "falloir",
    "mettre",
    "aller",
    "agir",
    "savoir",
    "répondre",
    "attendre",
    "permettre",
    "croire",
    "ici",
    "oui",
    "non",
    "beaucoup",
    "ensemble",
    "bien",
    "toujours",
    "aujourd'hui",
    "maintenant",
    "ainsi",
    "plus",
    "moins",
    "année",
    "suite",
    "jour",
    "nouveau",
    "nouvelle",
    "fois",
    "moment",
    "cadre",
    "objet",
    "point",
    "particulier",
    "notamment",
}

DISCOURSE_MARKERS_NORMALIZED = {strip_accents(w).lower() for w in DISCOURSE_MARKERS}


def clean_archelec_manifesto(text: str) -> str:
    """
    Sanitizes manifestos to create a stationary feature space by neutralizing
    OCR noise, geographical leakage, protecting acronyms, and tracking negation.
    """
    if not isinstance(text, str):
        return ""

    # 0. Acronym Stitching (Execute BEFORE spaCy to bypass length thresholds)
    text = re.sub(r"\bP\.?\s*S\.?\b", "parti_socialiste", text, flags=re.IGNORECASE)
    text = re.sub(r"\bP\.?\s*C\.?\b", "parti_communiste", text, flags=re.IGNORECASE)
    text = re.sub(r"\bR\.?\s*P\.?\s*R\.?\b", "rpr", text, flags=re.IGNORECASE)
    text = re.sub(r"\bU\.?\s*D\.?\s*F\.?\b", "udf", text, flags=re.IGNORECASE)

    # 1. OCR Hyphenation & Archival Noise Correction
    text = re.sub(r"-\s+", "", text)
    text = re.sub(r"(?i)sciences\s*po\s*/\s*fonds\s*cevipof", "", text)
    text = re.sub(r"(?i)imprimer[il]e\s+[^\n]+", "", text)
    text = re.sub(r"(?i)vu\s+les\s+candidats?", "", text)

    doc = nlp(text)
    sanitized_tokens = []
    negation_active = False  # State tracker for semantic inversion

    for token in doc:
        # A. Reset negation state on clause boundaries to prevent contamination
        if token.is_punct:
            negation_active = False
            continue

        # B. Trigger negation state
        if token.lemma_.lower() in ["ne", "n'"]:
            negation_active = True
            continue

        # C. Filter Spatial/Biographical Data Leakage
        if token.ent_type_ in ["LOC", "PER"]:
            continue

        # D. Morphosyntactic Filtering
        if token.pos_ not in ["NOUN", "ADJ", "VERB", "PROPN"]:
            continue

        lemma_norm = strip_accents(token.lemma_.lower())

        # E. Purge Auxiliaries, Gazetteer, and Null-Variance Markers
        if lemma_norm in ["etre", "avoir"]:
            continue

        if lemma_norm in GAZETTEER or lemma_norm in DISCOURSE_MARKERS_NORMALIZED:
            continue

        if token.is_stop or token.is_space:
            continue

        # F. Residual OCR Scoria Filter
        if len(lemma_norm) < 3 or not lemma_norm.isalpha():
            continue

        # G. Apply ideological inversion prefix to the FIRST surviving semantic token
        if negation_active:
            lemma_norm = "ne_" + lemma_norm
            negation_active = False

        sanitized_tokens.append(lemma_norm)

    return " ".join(sanitized_tokens)

In [ ]:
manifesto_eg = main_parties_metadata_df.index[440]
with open(f"../data/manifestos/1981/legislatives/{manifesto_eg}.txt", "r") as f:
    print(f.read())

ensemble
le 14 juin confirmons l'espoir du 10 mai
1er circonscription Montpellier-Lunel
République Française - Département de l'Hérault Elections législatives de juin 1981
GEORGES FRÊCHE
Maire de Montpellier Professeur à la Faculté de Droit
Suppléant : Elie RAUZIER Maire de Lunel, ingénieur
Parti socialiste
Au soir du 10 Mai François Mitterrand, notre Président, déclarait : « nous avons tant de choses à faire ensemble ». Donnons-lui la majorité qui lui permettra de les réaliser. Donnons à la France les députés socialistes dont elle a besoin. L'élan est donné. La France repart, la France revit, la France retrouve sa jeunesse. Ensemble continuons notre chemin avec le Président, avec la majorité de la France. Pour que s'expriment enfin les forces de la vie, les forces de la création, les forces du changement. Ensemble nous allons relancer l'économie. Nous allons donner la priorité à l'emploi. Nous allons construire une France où chacun aura sa place. Ensemble le 14 Juin, confirmons l'imme

In [70]:
manifesto_eg = main_parties_metadata_df.index[440]
with open(f"../data/manifestos/1981/legislatives/{manifesto_eg}.txt", "r") as f:
    text = f.read()
cleaned_text = clean_archelec_manifesto(text)

In [71]:
import textwrap

# Wraps the text cleanly at 80 characters per line
print(textwrap.fill(cleaned_text, width=80))

juin confirmer mai juin george professeur faculte droit ingenieur parti
socialiste soir mai declarer chose donnons majorite realiser donner socialiste
besoin donner repartir revoir retrouver jeunesse continuer president majorite
exprimer creation changement relancer economie donner priorite emploi construire
juin confirmer immense mai tranquillite construire avenir justice social smic
relever tva abaisser produit necessite integration handicape societe dignite
retrouver personne agee avenir liberer droit travailleur droit egal femme statut
femme commercant agriculteur justice independant emploi relance economique
croissance investissement favoriser epargne proteger developpement commercant
artisan paix respecter monde ordre economique mondial solidarite tiers monde
agriculture reforme politique garantie agriculteur face elargissement statut
femme agriculteur garantie revenu viticulteur producteur fruit legume emploi
aide pme pmi vivre travailler pays emploi femme decentralisation regio

- Improve gazetter with INSEE Database COG
- Improve parties label with conseil d'état analysis 